# Steam EDA\nExploratory analysis for the Steam recommendation dataset.

In [ ]:
from pathlib import Path\nimport sys\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\n\nROOT = Path.cwd().parent\nif str(ROOT) not in sys.path:\n    sys.path.append(str(ROOT))\n\nfrom src.data_loader import (\n    build_interaction_matrices,\n    filter_recommendations,\n    load_steam_data,\n    print_dataset_statistics,\n)\n\nsns.set_theme(style='whitegrid')

In [ ]:
DATA_DIR = ROOT / 'data'\nrecommendations, games, users = load_steam_data(DATA_DIR)\n\nprint('Raw recommendations rows:', len(recommendations))\nprint('Games rows:', len(games))\nprint('Users rows:', len(users))\n\nfiltered = filter_recommendations(recommendations, min_user_reviews=5, min_game_reviews=50)\ninteraction_data = build_interaction_matrices(filtered)\n\nprint_dataset_statistics(interaction_data.binary_matrix, title='Binary interaction matrix')\nprint_dataset_statistics(interaction_data.hours_matrix, title='Hours interaction matrix')

In [ ]:
reviews_per_user = filtered.groupby('user_id').size()\nreviews_per_game = filtered.groupby('app_id').size()\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\naxes[0].hist(reviews_per_user, bins=100)\naxes[0].set_yscale('log')\naxes[0].set_xscale('log')\naxes[0].set_title('Reviews per user (log-log)')\naxes[0].set_xlabel('Reviews')\naxes[0].set_ylabel('Count')\n\naxes[1].hist(reviews_per_game, bins=100)\naxes[1].set_yscale('log')\naxes[1].set_xscale('log')\naxes[1].set_title('Reviews per game (log-log)')\naxes[1].set_xlabel('Reviews')\naxes[1].set_ylabel('Count')\n\nplt.tight_layout()\nplt.show()

In [ ]:
binary_matrix = interaction_data.binary_matrix.tocsr()\nm, n = binary_matrix.shape\n\nrng = np.random.default_rng(42)\nsample_users = rng.choice(m, size=min(500, m), replace=False)\nsample_games = rng.choice(n, size=min(500, n), replace=False)\n\nsubmatrix = binary_matrix[sample_users][:, sample_games].toarray()\n\nplt.figure(figsize=(8, 6))\nsns.heatmap(submatrix, cmap='mako', cbar=False)\nplt.title('Random 500x500 submatrix sparsity heatmap')\nplt.xlabel('Games')\nplt.ylabel('Users')\nplt.show()

In [ ]:
output_path = DATA_DIR / 'interactions_filtered.csv'\nfiltered.to_csv(output_path, index=False)\nprint(f'Saved filtered interactions to: {output_path}')